## **Conexión de Snowflake y Python.**

Para conectar Snowflake con Pyrthon se requiere de la libreria snowflake-connector-python.

A continuación nos centraremos en la conexión a Snowflake.
La conexión se cea utilizando snowflake.connector.connect(), proporcionando las credenciales de  acceso y la informacion de nuestra cuenta de Snowflake.

Tambien se utilizara el método conn.cursos() y su objetivo es ejecutar consultas SQL y obtener los resultados que deseamos.

In [1]:
import snowflake.connector
import os
import configparser

# Conexión a Snowflake
config_path = os.path.join(os.environ['USERPROFILE'], '.snowsql', 'config')
config = configparser.ConfigParser()
config.read(config_path)

# Se obtiene la información desde nuestro archivo de configuración previamente creado.
# El archivo de configuración contiene el nombre de la cuenta, el nimbre de usuario y la contraseña de nuestra cuenta de Snowflake.
try:
    account = config['connections.example']['accountname']
    user = config['connections.example']['username']
    password = config['connections.example']['password']
except KeyError as e:
    print(f'Error: {e}')

# Se realiza la conexión a Snowflake utilizando la información obtenida del archivo de configuración.
try:
    conn = snowflake.connector.connect(
        user = user,
        password = password,
        account = account
    )
    print('Conexión exitosa.')
except Exception as e:
    print(f'Error: {e}')

Conexión exitosa.


## **Ejecución de consultas SQL desde Python.**

In [ ]:
# En esta celda creamos un cursor para consultas sql y seleccionamos el warehouse y la base de datos que vamos a utilizar.

cur = conn.cursor()

cur.execute('USE WAREHOUSE COMPUTE_WH;')

cur.execute('USE DATABASE DEVMOON_SAMPLE;')


In [ ]:
# Consulta para obtener las tiendas que tuvieron ventas mayores al promedio de ventas de todas las tiendas del mes de febrero.
"""
Se utiliza una subconsulta que calcula el promedio de ventas de las tiendas en febrero y se compara con las ventas de cada tienda.
Se utiliza un CTE para calcular las ventas de cada tienda en febrero y se hace un LEFT JOIN con la tabla de tiendas para obtener el 
nombre de la ciudad de cada tienda.
Finalmente, se filtran las tiendas que tuvieron ventas mayores al promedio y se muestran en la salida.
"""

sql = """
WITH SALES_STORE AS (
    SELECT STORE, SUM(WEEKLY_SALES) AS WEEKLY_SALES
    FROM SALES
    WHERE EXTRACT(MONTH FROM DDATE) = 2
    GROUP BY STORE
)
SELECT ST.STORE, S.CITY
FROM SALES_STORE ST
LEFT JOIN STORES S USING (STORE)
WHERE ST.WEEKLY_SALES > (SELECT AVG(WEEKLY_SALES) FROM SALES_STORE);
"""
cur.execute(sql)
result = cur.fetchall()
for row in result:
    print(row)

(1, 'Honolulu')
(2, 'Tulsa')
(4, 'Madison')
(6, 'Charlottesville')
(10, 'Montgomery')
(11, 'Las Vegas')
(12, 'Newark')
(13, 'Fullerton')
(14, 'Mobile')
(18, 'Phoenix')
(19, 'Clearwater')
(20, 'Norwalk')
(23, 'El Paso')
(24, 'Oklahoma City')
(27, 'Whittier')
(28, 'Louisville')
(31, 'Cheyenne')
(32, 'Arlington')
(39, 'Peoria')
(41, 'Columbus')


In [ ]:
# Consulta para obtener las 3 ciudades con mayor promedio de ventas en días festivos.
"""
Se utiliza un INNER JOIN entre las tablas de ventas y tiendas para obtener la ciudad de cada tienda.
Se filtran las ventas que ocurrieron en días festivos y se calcula el promedio de ventas por ciudad.
Finalmente, se ordenan las ciudades por promedio de ventas en días festivos y se seleccionan las 3 primeras.
"""

sql = """
SELECT ST.CITY, AVG(SL.WEEKLY_SALES) AS AVG_WKLY_SALES
FROM DEVMOON_SAMPLE.PUBLIC.SALES AS SL
INNER JOIN DEVMOON_SAMPLE.PUBLIC.STORES AS ST ON SL.STORE = ST.STORE
WHERE SL.HOLIDAY_FLAG = 1
GROUP BY ST.CITY
ORDER BY AVG(SL.WEEKLY_SALES) DESC
LIMIT 3;
"""
cur.execute(sql)
result = cur.fetchall()
for row in result:
    print(row)

('Norwalk', 2249035.081)
('Madison', 2243102.6240000003)
('Mobile', 2120582.998)


In [ ]:
# Consulta para obtener el promedio de temperatura por tipo de tienda en días no festivos.
"""
Se utiliza un LEFT JOIN entre las tablas de ventas y la información de las tiendas para obtener el tipo de tienda.
Se filtran las ventas que ocurrieron en días no festivos y se calcula el promedio de temperatura por tipo de tienda.
Finalmente, se agrupan los resultados por tipo de tienda y se muestran en la salida.
"""

sql = """
SELECT SIF.TYPE, AVG(S.TEMPERATURE) AS AVG_TEMPERATURE
FROM SALES S
LEFT JOIN STORES_INFO SIF USING(STORE)
WHERE S.HOLIDAY_FLAG = 0
GROUP BY SIF.TYPE;
"""
cur.execute(sql)
result = cur.fetchall()
for row in result:
    print(row)

('A', 61.820857826384135)
('B', 58.554135338345866)
('C', 68.28106516290727)


In [ ]:
# Consulta para obtener la tienda con mayor desempleo.
"""
Se utiliza un LEFT JOIN entre las tablas de ventas y la información de las tiendas para obtener la ciudad y dirección de cada tienda.
Se ordenan los resultados por la columna de desempleo en orden descendente y se selecciona la primera fila, 
que corresponde a la tienda con mayor desempleo.
"""

sql = """
SELECT ST.STORE, ST.CITY, ST.ADDRESS
FROM SALES SL
LEFT JOIN STORES ST USING (STORE)
ORDER BY SL.UNEMPLOYMENT DESC
LIMIT 1;
"""
cur.execute(sql)
result = cur.fetchall()
for row in result:
    print(row)

(12, 'Newark', '2 Bobwhite Pass')


## **Extracción de datos a dataframes de Pandas.**

In [ ]:
# En esta celda se utiliza la función fetch_pandas_all() para obtener los resultados de la consulta en un DataFrame de pandas.

import pandas as pd
pd = cur.fetch_pandas_all()
pd

,STORE,CITY,ADDRESS
0,12,Newark,2 Bobwhite Pass


In [ ]:
# Se utiliza la función info() de pandas para mostrar información sobre el DataFrame, incluyendo el número de entradas, columnas, tipos de datos y memoria utilizada.

pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   STORE    1 non-null      int8  
 1   CITY     1 non-null      object
 2   ADDRESS  1 non-null      object
dtypes: int8(1), object(2)
memory usage: 149.0+ bytes
